# Fine-tune Qwen3.5-35B-A3B with Unsloth on RunPod

Fine-tunes Qwen3.5-35B-A3B (MoE: 35B total, 3B active per forward pass) for:
- **LDA** — Legal Document Anonymization (PII detection and replacement)
- **Legal Drafting** — Board resolutions, memos, engagement letters
- **Agentic Reasoning** — Multi-step task planning and instruction following

**RunPod Setup:**
1. Create GPU Pod → **A100 80GB SXM** (~$1.64/hr)
2. Container disk: **300 GB** (model + export intermediates)
3. Template: **RunPod PyTorch 2.x** (or any with CUDA 12+)
4. Start pod → Connect → **Jupyter Lab**
5. Upload this notebook + `train.jsonl` + `valid.jsonl` via Jupyter file browser

**Training:** 8-bit LoRA (A100 80GB has enough VRAM). Export as Q5_K_M GGUF (~22GB) for 32GB Mac Mini.

## 1. Setup Environment

In [ ]:
import os
# MUST be set before ANY imports — disables Xet storage backend (known stalling bug)
# See: https://github.com/huggingface/xet-core/issues/483
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"

In [ ]:
%%capture
!pip uninstall hf_xet hf-xet -y 2>/dev/null
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Download Model

Xet storage backend disabled (known stalling bug). Downloads via standard HTTP with resume support.

In [ ]:
# Download model (Xet disabled in cell above, uses standard HTTP)
# Re-run this cell if it stalls — it resumes from where it left off
from huggingface_hub import snapshot_download
import time

for attempt in range(5):
    try:
        snapshot_download(
            repo_id="unsloth/Qwen3.5-35B-A3B",
            resume_download=True,
            max_workers=1,  # Single-threaded avoids connection issues
        )
        print("Download complete!")
        break
    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}, retrying in 10s...")
        time.sleep(10)

## 3. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

# Try 8-bit first (better quality), fall back to 4-bit if OOM
# 8-bit: ~39GB weights, should fit A100 80GB with gradient checkpointing
# 4-bit: ~20GB weights, proven to work on G4 95GB (loss 1.3385)
TRY_8BIT = True

try:
    if TRY_8BIT:
        print("Attempting 8-bit loading...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/Qwen3.5-35B-A3B",
            max_seq_length=4096,
            dtype=None,
            load_in_4bit=False,
            load_in_8bit=True,
        )
        print("8-bit model loaded successfully!")
    else:
        raise RuntimeError("Skipping to 4-bit")
except Exception as e:
    print(f"8-bit failed: {e}")
    print("Falling back to 4-bit...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3.5-35B-A3B",
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
    )
    print("4-bit model loaded successfully!")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Upload Training Data

Upload `train.jsonl` and `valid.jsonl` — each line is `{"messages": [{"role": ..., "content": ...}, ...]}`.

In [ ]:
import json

# Upload train.jsonl and valid.jsonl via Jupyter Lab file browser (drag & drop)
# They should be in the same directory as this notebook

# Verify data format
with open("train.jsonl") as f:
    first = json.loads(f.readline())
    print(f"First example roles: {[m['role'] for m in first['messages']]}")
    print(f"System prompt preview: {first['messages'][0]['content'][:100]}...")

# Count examples
train_count = sum(1 for _ in open("train.jsonl"))
valid_count = sum(1 for _ in open("valid.jsonl"))
print(f"\nTrain: {train_count} examples, Valid: {valid_count} examples")

## 4. Prepare Dataset

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Set chat template for Qwen3.5
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",  # Qwen3/3.5 uses same template
)

# Load JSONL datasets
dataset = load_dataset("json", data_files={"train": "train.jsonl", "validation": "valid.jsonl"})

def format_chat(example):
    """Convert messages to chat template format"""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=["messages"])

# Show a formatted example
print("=== Formatted example (first 500 chars) ===")
print(dataset["train"][0]["text"][:500])

## 5. Configure Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2,
    packing=True,  # Pack short examples together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,  # Larger effective batch for 8-bit
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

## 6. Train

In [ ]:
# Show memory stats before training
gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name}")
print(f"VRAM: {gpu_stats.total_memory / 1024**3:.1f} GB")
print(f"Compute: {gpu_stats.major}.{gpu_stats.minor}")

# Start training
trainer_stats = trainer.train()

# Print results
print(f"\nTraining complete!")
print(f"Total steps: {trainer_stats.global_step}")
print(f"Final loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")

## 7. Test the Model

Run inference on sample LDA and drafting prompts to verify the fine-tune.

In [ ]:
FastLanguageModel.for_inference(model)

# Test LDA
messages_lda = [
    {"role": "system", "content": "You are a legal document anonymizer. Identify all personally identifiable information (PII) in the document and replace each instance with a typed placeholder. Use these placeholder types: {PERSON_N} for personal names, {ORG_N} for company/organization names, {ADDRESS_N} for street addresses, {DATE_N} for specific dates, {AMOUNT_N} for monetary amounts, {PHONE_N} for phone numbers, {EMAIL_N} for email addresses, {TITLE_N} for job titles. Maintain consistent numbering. After the anonymized text, include a mapping section."},
    {"role": "user", "content": "John Smith, CEO of Acme Corp, located at 123 Main Street, New York, NY 10001, will receive an annual salary of $500,000 effective January 1, 2025. His emergency contact is Jane Smith at (212) 555-0199."},
]

inputs = tokenizer.apply_chat_template(messages_lda, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=500, temperature=0.3)
print("=== LDA Test ===")
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

# Test Drafting
messages_draft = [
    {"role": "system", "content": "You are a legal drafting assistant for a New York-based cross-border law firm."},
    {"role": "user", "content": "Draft a brief board resolution authorizing the CEO to execute a consulting agreement with an outside technology firm, not to exceed $50,000."},
]

inputs2 = tokenizer.apply_chat_template(messages_draft, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs2 = model.generate(input_ids=inputs2, max_new_tokens=1000, temperature=0.3)
print("\n=== Drafting Test ===")
print(tokenizer.decode(outputs2[0][inputs2.shape[-1]:], skip_special_tokens=True))

### Troubleshooting

If the LDA test shows incorrect entity boundaries or wrong placeholder types, try:
1. Increase epochs to 5 (`num_train_epochs=5` in training config)
2. Increase LoRA rank to 32 (`r=32` in model config)
3. Lower learning rate to `1e-5` if loss is unstable

## 8. Save and Export to GGUF

With 300GB disk on RunPod, the one-shot export works fine (no staged conversion needed).

In [ ]:
import shutil

# Save LoRA adapter (backup)
model.save_pretrained("qwen3.5-legal-lora")
tokenizer.save_pretrained("qwen3.5-legal-lora")

# Export to GGUF — Q5_K_M fits 32GB Mac Mini while preserving quality
model.save_pretrained_gguf(
    "qwen3.5-legal-gguf",
    tokenizer,
    quantization_method="q5_k_m",
)

# Show results
import os
print("Exported to GGUF!")
print("Files in qwen3.5-legal-gguf/:")
for f in os.listdir("qwen3.5-legal-gguf"):
    size = os.path.getsize(f"qwen3.5-legal-gguf/{f}") / 1024**3
    print(f"  {f}: {size:.1f} GB")

total, used, free = shutil.disk_usage("/")
print(f"\nDisk: {free / 1024**3:.1f} GB free")

## 9. Download GGUF

Download the GGUF file from RunPod's Jupyter file browser, or use `scp` from the terminal.

In [ ]:
import glob, os

# Find the GGUF file
gguf_files = glob.glob("qwen3.5-legal-gguf/*.gguf")
for f in gguf_files:
    size = os.path.getsize(f) / 1024**3
    print(f"Ready to download: {f} ({size:.1f} GB)")

# Also zip the LoRA adapter as backup
!zip -r qwen3.5-legal-lora.zip qwen3.5-legal-lora/

print("\n=== Download options ===")
print("Option A: Download via Jupyter file browser (click the file)")
print("Option B: From your Mac, run:")
print("  scp -P <RUNPOD_SSH_PORT> root@<RUNPOD_IP>:~/qwen3.5-legal-gguf/*.gguf ~/Downloads/")
print("\n=== After downloading, on your Mac Mini ===")
print("1. scp qwen3.5-legal-q5_k_m.gguf claptrap@ClapTraps-Mac-mini.local:~/finetune/")
print("2. Create Modelfile:")
print('   FROM ./qwen3.5-legal-q5_k_m.gguf')
print('   PARAMETER temperature 0.3')
print('   PARAMETER num_ctx 8192')
print("3. cd ~/finetune && ollama create qwen3.5-legal -f Modelfile")
print("4. Update openclaw.json Associate agent model to 'ollama/qwen3.5-legal'")

## 10. Optional: 4-bit Training (Lower VRAM)

If your RunPod GPU has less than 80GB VRAM (e.g. A100 40GB, RTX 4090), switch to 4-bit:

Changes needed in Cell 7:
```python
load_in_4bit=True,
load_in_8bit=False,  # remove this line
```

Note: 4-bit training loses some precision for entity boundary detection in LDA tasks. 8-bit (default) is recommended when VRAM allows.